In [1]:
# 코사인 유사도
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(a,b):
    return dot(a,b) / (norm(a)*norm(b))

vec1 = np.array([0,1,1,1])
vec2 = np.array([1,0,2,1])
vec3 = np.array([2,0,4,2])

print('vec1과 vec2의 유사도:',cos_sim(vec1,vec2))
print('vec1과 vec3의 유사도:',cos_sim(vec1,vec3))
print('vec2과 vec3의 유사도:',cos_sim(vec2,vec3))

vec1과 vec2의 유사도: 0.7071067811865476
vec1과 vec3의 유사도: 0.7071067811865476
vec2과 vec3의 유사도: 1.0000000000000002


In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
from langchain_openai import OpenAI, OpenAIEmbeddings

load_dotenv("./.env")
api_key = os.getenv("OPENAI_API_KEY")

c:\Users\Dohy\Desktop\dohy\RAG_master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embeddings = OpenAIEmbeddings(model='text-embedding-ada-002')
query_result = embeddings.embed_query("저는 배가 고파요")
print(query_result)
print(len(query_result))

[-0.016708848997950554, -0.021796436980366707, 0.01521017961204052, -0.02723897434771061, -0.0367831327021122, 0.011857893317937851, -0.034574564546346664, -0.006744012236595154, -0.023912979289889336, -0.016840310767292976, -0.008111219853162766, 0.010871926322579384, -0.010622148402035236, -0.022598357871174812, 0.011240020394325256, -0.00493640685454011, 0.012087952345609665, -0.0028708064928650856, 0.00810464657843113, -0.01610412262380123, 0.001368028810247779, -0.014473991468548775, 0.018207518383860588, -0.012705824337899685, 0.003214251482859254, 0.006418643519282341, 0.00539652444422245, -0.019062023609876633, -0.009215502068400383, -0.0017237984575331211, 0.03449568897485733, -0.01651165634393692, 1.0655629921529908e-05, 0.0030959355644881725, 0.007375031244009733, -0.005560852121561766, -0.006428502965718508, 0.003424591151997447, -0.003828837303444743, -0.0022085653617978096, -0.022480040788650513, -0.010319785214960575, 0.011358336545526981, -0.024662313982844353, 0.013211

In [4]:
data=[
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요',
]

df = pd.DataFrame(data,columns=['text'])
df

,text
0,주식 시장이 급등했어요
1,시장 물가가 올랐어요
2,전통 시장에는 다양한 물품들을 팔아요
3,부동산 시장이 점점 더 복잡해지고 있어요
4,저는 빠른 비트를 좋아해요
5,최근 비트코인 가격이 많이 변동했어요


In [5]:
def get_embedding(text):
    return embeddings.embed_query(text)

df['embedding'] = df.apply(lambda row: get_embedding(row.text), axis=1)
df

,text,embedding
0,주식 시장이 급등했어요,"[-0.013810144737362862, -0.03508242964744568, ..."
1,시장 물가가 올랐어요,"[-0.00015702718519605696, -0.03539642691612243..."
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.009272306226193905, -0.01461657416075468, ..."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[-0.017441052943468094, -0.0031565295066684484..."
4,저는 빠른 비트를 좋아해요,"[-0.03796839341521263, -0.013386794365942478, ..."
5,최근 비트코인 가격이 많이 변동했어요,"[-0.01749446988105774, -0.021561449393630028, ..."


In [6]:
def return_answer_candidate(df, query):
    query_embedding = get_embedding(query)
    df['similarity'] = df.embedding.apply(lambda x: cos_sim(np.array(x), np.array(query_embedding)))
    top_three_doc = df.sort_values('similarity', ascending=False).head(3)
    return top_three_doc

sim_result = return_answer_candidate(df, '과일 값이 비싸다')
sim_result

,text,embedding,similarity
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.009272306226193905, -0.01461657416075468, ...",0.819454
1,시장 물가가 올랐어요,"[-0.00015702718519605696, -0.03539642691612243...",0.809867
0,주식 시장이 급등했어요,"[-0.013810144737362862, -0.03508242964744568, ...",0.803069


In [7]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
data=[
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요',
]

df = pd.DataFrame(data,columns=['text'])
df['embedding'] = df['text'].apply(get_embedding)
df

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 96753.56it/s]


,text,embedding
0,주식 시장이 급등했어요,"[-0.028310665860772133, -0.002016811165958643,..."
1,시장 물가가 올랐어요,"[0.003978129010647535, 0.02656502276659012, -0..."
2,전통 시장에는 다양한 물품들을 팔아요,"[0.00010141320672119036, 0.00058088602963835, ..."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[-0.007726868148893118, 0.021975180134177208, ..."
4,저는 빠른 비트를 좋아해요,"[-0.01941235363483429, 0.012223081663250923, -..."
5,최근 비트코인 가격이 많이 변동했어요,"[-0.02639133110642433, -0.004527537617832422, ..."


In [8]:
sim_result = return_answer_candidate(df,'과일 값이 비싸다')
sim_result

,text,embedding,similarity
1,시장 물가가 올랐어요,"[0.003978129010647535, 0.02656502276659012, -0...",0.642563
5,최근 비트코인 가격이 많이 변동했어요,"[-0.02639133110642433, -0.004527537617832422, ...",0.604495
0,주식 시장이 급등했어요,"[-0.028310665860772133, -0.002016811165958643,...",0.575970


In [9]:
# WebBaseLoader
from langchain_community.document_loaders import WebBaseLoader
import os
# 사용자 에이전트 설정
os.environ['USER_AGENT'] = "MyApp/1.0 (Custom LangChain Application)" # 서버가 클라이언트 요청을 더 쉽게 식별하고 추적하기 위해 USER_AGENT 환경 변수 설정, *User-Agent는 HTTP 요청을 보낼 때 함께 딸려가는 정보

# 단일 URL 초기화
loader = WebBaseLoader("https://docs.smith.langchain.com")

# 다중 URL 초기화
loader_multiple_pages = WebBaseLoader(
    ['https://python.langchain.com/docs/introduction/',
     'https://langchain-ai.github.io/langgraph']
)

# 단일 문서 로드 및 메타데이터 확인
single_doc = loader.load()
print(single_doc[0].metadata)

# 다중 문서 로드 및 첫 번쨰 문서의 페이지 콘텐츠 확인
docs = loader_multiple_pages.load()
print(docs[0].page_content)


C:\Users\Dohy\AppData\Local\Temp\ipykernel_2672\1134615305.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


{'source': 'https://docs.smith.langchain.com', 'title': 'LangSmith overview - Docs by LangChain', 'language': 'en'}
LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create

In [10]:
%%time
# PDF 로더
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, PDFPlumberLoader

# PDF 파일 로더 초기화
loader = PyPDFLoader("./Data/2024_KB_부동산_보고서_최종.pdf")

# PDF 파일 로드 및 페이지 분할
pages = loader.load_and_split() # RecursiveCharacterTextSplitter를 사용하여 PDF를 청클 분할(청크 크기(chunk_size), 중복 범위(chunk_overlap), 분할 우선순위 문자(separators)를 조정하여 더 세밀하게 분할할 수 있음)
print('청크 수: ', len(pages))
pages[10]

청크 수:  83
CPU times: total: 4.27 s
Wall time: 4.27 s


Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 11, 'page_label': '12'}, page_content='5 \n2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 \n \n표Ⅰ-1. 공급 확대 정책 주요 내용 \n날짜 주요 내용 \n2023년 9월 \n공공 부문 공급물량 확대: 3기 신도시를 포함한 공급물량 확대 및 조기 공급 방안 마련 \n사업 여건 개선: 전매 제한 완화 및 규제 정상화, 조기 인허가 인센티브 및 절차 개선, 공사비 증액 \n기준 마련 및 인력 확충, 분양 사업의 임대 사업 전환 촉진 \n원활한 자금 지원: PF대출 보증 확대, 중도금 대출 지원 \n비아파트 자금 조달 지원 및 규제 개선 \n도심 공급 기반 확충: 정비사업 절차 및 소규모 사업 사업성 개선 \n2024년 1월 \n도심 공급 확대: 재건축·재개발 패스트트랙 도입 및 재건축 부담금 합리화, 1기 신도시 재정비사업의 \n신속하고 내실 있는 계획 수립, 소규모 정비·도심 복합 사업 속도 개선 및 자금 지원 강화 \n다양한 유형의 주택 공급 확대: 도시·건축 규제 완화 및 세제·금융 지원, 등록 임대 사업 여건 개선 \n및 기업형 장기임대 활성화, 신축 매입 약정 확대, 전세 사기 예방 및 피해 지원 \n신도시 등 공공주택 공급: 2024년 공공주택 14만 호 이상 공급, 신도시 조성 속도 제고 \n건설 경기 활력 회복: PF대출 지원 및 유동성 지원, 공공 임대 참여 지분 조기 매각, 민간 사업

In [11]:
print(pages[10].metadata)
print(pages[10].page_content)

{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 11, 'page_label': '12'}
5 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
표Ⅰ-1. 공급 확대 정책 주요 내용 
날짜 주요 내용 
2023년 9월 
공공 부문 공급물량 확대: 3기 신도시를 포함한 공급물량 확대 및 조기 공급 방안 마련 
사업 여건 개선: 전매 제한 완화 및 규제 정상화, 조기 인허가 인센티브 및 절차 개선, 공사비 증액 
기준 마련 및 인력 확충, 분양 사업의 임대 사업 전환 촉진 
원활한 자금 지원: PF대출 보증 확대, 중도금 대출 지원 
비아파트 자금 조달 지원 및 규제 개선 
도심 공급 기반 확충: 정비사업 절차 및 소규모 사업 사업성 개선 
2024년 1월 
도심 공급 확대: 재건축·재개발 패스트트랙 도입 및 재건축 부담금 합리화, 1기 신도시 재정비사업의 
신속하고 내실 있는 계획 수립, 소규모 정비·도심 복합 사업 속도 개선 및 자금 지원 강화 
다양한 유형의 주택 공급 확대: 도시·건축 규제 완화 및 세제·금융 지원, 등록 임대 사업 여건 개선 
및 기업형 장기임대 활성화, 신축 매입 약정 확대, 전세 사기 예방 및 피해 지원 
신도시 등 공공주택 공급: 2024년 공공주택 14만 호 이상 공급, 신도시 조성 속도 제고 
건설 경기 활력 회복: PF대출 지원 및 유동성 지원, 공공 임대 참여 지분 조기 매각, 민간 사업장 
공공 인수, 재정 조기 집행 및 민자 사업 확대 
 
자료: 국토교통부 보도자료 요약 

In [12]:
%%time

loader = PyMuPDFLoader('./Data/2024_KB_부동산_보고서_최종.pdf')
pages = loader.load_and_split()
print('청크의 수: ', len(pages))
pages[10]

청크의 수:  83
CPU times: total: 250 ms
Wall time: 260 ms


Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'file_path': './Data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'format': 'PDF 1.5', 'title': 'Morning Meeting', 'author': '손은경', 'subject': '', 'keywords': '', 'moddate': '2024-03-04T15:30:01+09:00', 'trapped': '', 'modDate': "D:20240304153001+09'00'", 'creationDate': "D:20240304153001+09'00'", 'page': 11}, page_content='5 \n2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 \n표Ⅰ-1. 공급 확대 정책 주요 내용 \n날짜 \n주요 내용 \n2023년 9월 \n공공 부문 공급물량 확대: 3기 신도시를 포함한 공급물량 확대 및 조기 공급 방안 마련 \n사업 여건 개선: 전매 제한 완화 및 규제 정상화, 조기 인허가 인센티브 및 절차 개선, 공사비 증액 \n기준 마련 및 인력 확충, 분양 사업의 임대 사업 전환 촉진 \n원활한 자금 지원: PF대출 보증 확대, 중도금 대출 지원 \n비아파트 자금 조달 지원 및 규제 개선 \n도심 공급 기반 확충: 정비사업 절차 및 소규모 사업 사업성 개선 \n2024년 1월 \n도심 공급 확대: 재건축·재개발 패스트트랙 도입 및 재건축 부담금 합리화, 1기 신도시 재정비사업의 \n신속하고 내실 있는 계획 수립, 소규모 정비·도심 복합 사업 속도 개선 및 자금 지원 강화 \n다양한 유형의 주택 공급 확대: 도시·건축 규제 완화 및 세제·금융 지원, 등

In [13]:
%%time

loader = PDFPlumberLoader('./Data/2024_KB_부동산_보고서_최종.pdf')
pages = loader.load_and_split()
print('청크 수: ', len(pages))
pages[10]

청크 수:  83
CPU times: total: 8.5 s
Wall time: 8.53 s


Document(metadata={'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'file_path': './Data/2024_KB_부동산_보고서_최종.pdf', 'page': 11, 'total_pages': 84, 'Title': 'Morning Meeting', 'Author': '손은경', 'Creator': 'Microsoft® Word 2016', 'CreationDate': "D:20240304153001+09'00'", 'ModDate': "D:20240304153001+09'00'", 'Producer': 'Microsoft® Word 2016'}, page_content='2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망\n표Ⅰ-1. 공급 확대 정책 주요 내용\n날짜 주요 내용\n공공 부문 공급물량 확대: 3기 신도시를 포함한 공급물량 확대 및 조기 공급 방안 마련\n사업 여건 개선: 전매 제한 완화 및 규제 정상화, 조기 인허가 인센티브 및 절차 개선, 공사비 증액\n기준 마련 및 인력 확충, 분양 사업의 임대 사업 전환 촉진\n2023년 9월\n원활한 자금 지원: PF대출 보증 확대, 중도금 대출 지원\n비아파트 자금 조달 지원 및 규제 개선\n도심 공급 기반 확충: 정비사업 절차 및 소규모 사업 사업성 개선\n도심 공급 확대: 재건축·재개발 패스트트랙 도입 및 재건축 부담금 합리화, 1기 신도시 재정비사업의\n신속하고 내실 있는 계획 수립, 소규모 정비·도심 복합 사업 속도 개선 및 자금 지원 강화\n다양한 유형의 주택 공급 확대: 도시·건축 규제 완화 및 세제·금융 지원, 등록 임대 사업 여건 개선\n2024년 1월 및 기업형 장기임대 활성화, 신축 매입 약정 확대, 전세 사기 예방 및 피해 지원\n신도시 등 공공주택 공급: 2024년 공공주택 14만 호 이상 공급, 신도시 조성 속도 제고\n건설 경기 활력 회복: PF대출 지원 및 유동성 지원, 공공 임대 참여 지분 조기 매각, 민간 사업장\

In [14]:
%%time
# CSV 로더
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader

# CSV 파일 로더 초기화
loader = CSVLoader("./Data/서울시_부동산_실거래가_정보.csv", encoding='utf8')

# CSV 파일 로드 및 행 분할
documents = loader.load()
print('청크 수: ', len(documents))
documents[5].page_content

청크 수:  2001
CPU times: total: 15.6 ms
Wall time: 15.5 ms


'\ufeff접수연도: 2024\n자치구코드: 11410\n자치구명: 서대문구\n법정동코드: 11600\n법정동명: 창천동\n지번구분: \n지번구분명: \n본번: \n부번: \n건물명: \n계약일: 20241031\n물건금액(만원): 340000\n건물면적(㎡): 421.83\n토지면적(㎡): 284\n층: \n권리구분: \n취소일: \n건축년도: 2014\n건물용도: 단독다가구\n신고구분: 직거래\n신고한 개업공인중개사 시군구명: '

In [15]:
%%time
# UnstructuredCSVLoader
loader = UnstructuredCSVLoader('./Data/서울시_부동산_실거래가_정보.csv', mode='elements')
documents = loader.load()
print('청크 수', len(documents))
print(type(documents[0]))
print(str(documents[0].metadata)[:500])
print(documents[0].metadata.get('text_as_html'))
print(str(documents[0].page_content)[:500])


청크 수 1
<class 'langchain_core.documents.base.Document'>
{'source': './Data/서울시_부동산_실거래가_정보.csv', 'file_directory': './Data', 'filename': '서울시_부동산_실거래가_정보.csv', 'last_modified': '2026-06-12T16:31:33', 'text_as_html': "<table><tr><td>접수연도</td><td>자치구코드</td><td>자치구명</td><td>법정동코드</td><td>법정동명</td><td>지번구분</td><td>지번구분명</td><td>본번</td><td>부번</td><td>건물명</td><td>계약일</td><td>물건금액(만원)</td><td>건물면적(㎡)</td><td>토지면적(㎡)</td><td>층</td><td>권리구분</td><td>취소일</td><td>건축년도</td><td>건물용도</td><td>신고구분</td><td>신고한 개업공인중개사 시군구명</td></tr><tr><td>2024</td><td>11440</td><td>
<table><tr><td>접수연도</td><td>자치구코드</td><td>자치구명</td><td>법정동코드</td><td>법정동명</td><td>지번구분</td><td>지번구분명</td><td>본번</td><td>부번</td><td>건물명</td><td>계약일</td><td>물건금액(만원)</td><td>건물면적(㎡)</td><td>토지면적(㎡)</td><td>층</td><td>권리구분</td><td>취소일</td><td>건축년도</td><td>건물용도</td><td>신고구분</td><td>신고한 개업공인중개사 시군구명</td></tr><tr><td>2024</td><td>11440</td><td>마포구</td><td>11100</td><td>신수동</td><td>1</td><td>대지</td><td>228</td><td>3</td><td>마인빌</td><td>20241031</t

In [16]:
# 텍스트 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('./Data/2024_KB_부동산_보고서_최종.pdf')
pages = loader.load()

print('총 글자 수:', len(''.join([i.page_content for i in pages])))

총 글자 수: 82384


In [17]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

texts = text_splitter.split_documents(pages)
print('분할된 청크 수:',len(texts))
print(texts[1].page_content)
print(texts[2].page_content)

분할된 청크 수: 218
변수가 될 것이다. 기준금리 인하 시기와 인하 폭에 따라 주택 수요는 영향을 받을 수밖에 없기 때문이다. 
한편, 수요 위축으로 거래가 급감한 상황에서 실수요자 금융 지원, 관련 규제 완화 등 수요 회복을 위한 
정부 정책도 중요한 변수가 될 전망이다. 
 
 7대 이슈를 통해 바라보는 2024년 주택시장 
1 역대 최저 수준이 지속되고 있는 주택 거래 
 
주택 매매 거래는 2022년에 이어 2년 연속 침체. 총선 이후 정책 불확실성 해소와 금리 인하로 인한 회
복 가능성이 있으나 일부 지역 수요 쏠림 현상과 금리 인하 속도가 더딜 경우 회복세는 제한적일 전망 
2 주택공급 급격한 감소로 인한 공급 부족 가능성 
 
분양물량과 함께 장기적인 주택 공급 기반인 인허가물량까지 급감. 청약 수요 위축으로 분양 지연이 장기
화될 가능성이 높은 가운데 정부의 공급 정책 구체화가 매우 중요 
3 노후계획도시 특별법과 재건축 시장 영향
3 노후계획도시 특별법과 재건축 시장 영향 
 
2023년 말 국회를 통과한 「노후계획도시 특별법」 시행을 앞두고 당초 51곳이었던 대상 지역이 108곳으
로 확대. 단기적 효과는 제한적이나 사업진행이 구체화되면 시장 영향도 커질 것  
4 전세 수요 아파트 집중, 입주물량 부족으로 가격 상승 가능성 확대 
 
비아파트 전세 사기와 보증금 미반환 이슈 등으로 아파트로 전세 수요가 집중되는 가운데 수도권을 중심
으로 아파트 입주물량이 크게 감소함에 따라 다시 전세가격 상승세가 확대될 가능성이 존재 
5 주택 경기에 최대 화두로 부각되는 금리 인하 가능성 
 
금리 인하에 따른 매수 심리 회복에 대한 기대감이 높아지고 있지만 가계 부채 관리 강화와 경기 불확실
성 확대 등이 존재하는 시장에서 금리 인하 시기와 인하폭이 관건 
6 주택경기 위축에도 늘어나는 주택담보대출 
 
주택담보대출 증가세는 다소 둔화될 것으로 전망되나 가계부채에 대한 우려가 지속될 경우 매수세 회복을


In [18]:
print('1번 청크의 길이: ', len(texts[1].page_content))
print('2번 청크의 길이: ', len(texts[2].page_content))

1번 청크의 길이:  468
2번 청크의 길이:  491


In [19]:
# 의미 기반으로 분할하는 시맨틱 청킹
from langchain_experimental.text_splitter import SemanticChunker
loader = PyPDFLoader('./Data/2024_KB_부동산_보고서_최종.pdf')
pages = loader.load()

# SemanticChunker 초기화
text_splitter = SemanticChunker(embeddings=OpenAIEmbeddings())

# 텍스트를 의미 단위로 분할
chunks = text_splitter.split_documents(pages)

# 분할된 청크 수
print('분할된 청크 수:',len(chunks))

C:\Users\Dohy\AppData\Local\Temp\ipykernel_2672\1114872311.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


분할된 청크 수: 164


In [20]:
print(chunks[3])

page_content='2 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
 
 
Executive Summary 2 
 주택 매매시장 하락 전망 우세, 부동산 투자 선호도 하락 
• 2024년 주택 매매가격 지난해에 이어 올해도 하락 전망 우세, 높은 금리가 가장 큰 부담 
부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문 조사 결과, 2024년 전국 주택 
매매가격은 하락세가 이어질 것이라는 전망이 우세하였다. 다만 시장 급락에 대한 우려는 다소 완화된 
것으로 보인다.' metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 2, 'page_label': '3'}


In [21]:
print(chunks[4])

page_content='상승에 대한 전망이 2023년 대비 크게 증가했기 때문이다(전문가 21%p, 공인중개사 
17%p, PB 13%p  증가). 매매가격 하락요인으로는 높은 금리에 따른 이자부담이 가장 중요한 이유로 
조사되었다. • 2024년 주택 전세가격, 비수도권은 하락 전망이 우세하나 수도권 전망은 엇갈려  
2024년 전국 주택 전세가격에 대해 전문가의 53%, 공인중개사의 61%가 하락을 전망하였다. 하락폭에 
대해서는 3% 이하가 될 것이라는 의견이 많았다. 다만 2023년 하락 전망이 압도적으로 우세했던 것과 
달리 2024년에는 상승과 하락 전망에 격차가 크지 않았다. 지역별로 수도권에서는 주택 전세가격이 
상승과 하락 의견이 엇갈렸으나 비수도권에서는 하락 전망이 우세하였다. • 경기 회복을 위해서는 금리 인하와 대출 규제 완화가 중요 변수 
주택 경기 회복을 위해 필요한 핵심 정책으로 전문가와 공인중개사, PB 모두 금리 인하를 꼽았다. 다음으로 주택담보대출 지원, LTV·DSR 등 금융 규제 완화가 필요하다는 의견이 많았다. 특히 공인중개사 
그룹에서 금리와 대출 관련 정책의 필요성을 높게 평가하였다. 이는 현재 주택시장 침체가 수요 감소에 
따른 영향이 크며 수요 회복 여부가 향후 시장 흐름을 결정할 핵심 요인이라는 인식이 반영된 결과로 
풀이된다. • 투자 유망 부동산으로 아파트 분양과 신축 아파트, 재건축 
전문가, 공인중개사, PB 는 공통적으로 2024년 투자 유망 부동산으로 아파트 분양과 신축 아파트, 
재건축을 꼽았다. 아파트 분양과 신축 아파트는 지난해 비해 선호도가 높아졌으며, 재건축은 꾸준히 투자 
유망 부동산으로 주목을 받을 것으로 보인다. 전문가는 아파트 분양(28%), 공인중개사는 신축 
아파트(23%), PB는 재건축(27%)을 1순위 투자유망 부동산으로 꼽았다. • 고자산가는 투자 자산으로 예금과 채권을 선호, 부동산 경기 위축 및 고금리로 부동산 선호도는 하락  
PB 대상 설문조사에서 고자산가가 선호하는 

In [22]:
print(chunks[5])

page_content='3 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
  
 
 
Executive Summary 3 
 수도권 주택시장 전반적 침체, 강남권 등 선호 지역 상대적 강세 
과거와는 달리 높은 기준금리와 주택 매매가격, DSR 규제 등으로 매수자들의 구매 여력은 회복되지 
못하고 있으나, 2023년 이후 정부의 다양한 규제 완화로 매도자들의 기대 심리는 높아지고 있다. 재건축 
규제 완화, 광역급행철도(GTX, Great Train Express) 개통 등에 따라 지역별로 호재가 존재하며 부동산 
PF대출 부실, 전세보증금 반환 문제 등 전반적인 주택 경기 위축 요소들도 상존하고 있다. 구분 지역명 주요 이슈 
서울 
한강 
이남 
강남구 정부 규제 완화와 주택 수요 집중으로 긍정적 기대감 유지 
서초구 높은 분양가에도 청약 수요는 많으나 진행 중인 정비사업은 차별화 예상 
송파구 전세가격 하락에 영향을 줄 수 있는 대규모 단지 입주 예정 
양천구 재건축 사업 진행은 긍정적이나 토지거래허가구역으로 매매시장 위축 지속 
서울 
한강 
이북 
마포구 지역 내에서 매수세 변화가 나타나면서 지역간 매매가격 격차 확대 
용산구 한남뉴타운 사업 진행은 양호하나 여전히 큰 매도자와 매수자 간 간극 
성동구 대기 수요는 많으나 실제 매수 가담 여부가 중요한 요소로 작용할 것 
수도권 
위례 위례선 개통 예정과 위례신사선 착공 지연으로 긍정적·부정적 요인이 혼재 
과천 지식정보타운의 본격적인 입주와 교통 개발로 지역 확대 
동탄 교통 개선과 반도체 관련 수요보다 수도권 주택시장 및 금리 영향이 클 전망 
 
 상업용 부동산시장  
고금리와 경기 불확실성 확대 영향으로 2023년 상업용 부동산시장은 거래량이 크게 감소하고 평균 
매매가격 역시 하락하였다. 거래량 감소와 함께 매매가격이 하락하면서 2023년 상업용 부동산 거래총액은 
2022년 대비 34.8% 감소했다.' metadata={'producer': 'Microsoft® 

In [23]:
text_splitter = SemanticChunker(
    OpenAIEmbeddings(),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95,
)
chunks = text_splitter.split_documents(pages)
print("분할된 청크 수: ", len(chunks))

분할된 청크 수:  164


In [24]:
text_splitter = SemanticChunker(
    OpenAIEmbeddings(),
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=3,
)
chunks = text_splitter.split_documents(pages)
print("분할된 청크 수: ", len(chunks))

분할된 청크 수:  84


In [25]:
text_splitter = SemanticChunker(
    OpenAIEmbeddings(),
    breakpoint_threshold_type="interquartile",
    breakpoint_threshold_amount=1.5,
)
chunks = text_splitter.split_documents(pages)
print("분할된 청크 수: ", len(chunks))

분할된 청크 수:  142


In [26]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

loader = PyPDFLoader('./Data/2024_KB_부동산_보고서_최종.pdf')
pages = loader.load()
print("청크 수: ",len(pages)) # 말이 청크고, 페이지 수

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(pages)
print("분할된 청크 수: ", len(splits))

청크 수:  84
분할된 청크 수:  135


In [27]:
chunk_lengths= [len(chunk.page_content) for chunk in splits]
max_length = max(chunk_lengths)
min_length = min(chunk_lengths)
avg_length = sum(chunk_lengths) / len(chunk_lengths)

print('청크 최대 길이: ', max_length)
print('청크 최소 길이: ', min_length)
print('청크 평균 길이: ', avg_length)

청크 최대 길이:  1000
청크 최소 길이:  56
청크 평균 길이:  674.9481481481481


In [28]:
embedding_function = OpenAIEmbeddings()

persist_directory = "./chromaDB/"

# Chroma DB 생성 및 데이터 저장
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding_function,
    persist_directory=persist_directory
)
print("문서 수: ", vectordb._collection.count())

문서 수:  135


In [29]:
# Chroma DB 로드

embedding_function = OpenAIEmbeddings()

vectordb = Chroma(
    embedding_function = embedding_function,
    persist_directory = persist_directory
)

print('문서 수: ', vectordb._collection.count())

문서 수:  135


In [30]:
query = "수도권 주택 매매 전망"
top_three_docs = vectordb.similarity_search(query,k=2)
for i,doc in enumerate(top_three_docs, 1): # 시작 번호 1
    print(f"문서 {i}:")
    print(f"내용: {doc.page_content[:150]}...")
    print(f"메타 데이터: {doc.metadata}")
    print('-'*20)

문서 1:
내용: 8 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
실 등에 따른 주택 경기 불안을 이유로 매매를 망설이며 시세 대비 저렴한 매물에만 관심을 보였다. 결
국 매도자와 매수자 간 희망가격 차이로 인한 매매 거래 위축 현상은 2023년 거래 침체의 가...
메타 데이터: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'moddate': '2024-03-04T15:30:01+09:00', 'page_label': '15', 'page': 14, 'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'author': '손은경', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting'}
--------------------
문서 2:
내용: 18 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
그림Ⅰ-30. 수도권 입주물량과 전세가격 변동률 추이  그림Ⅰ-31. 기타지방 입주물량과 전세가격 변동률 추이 
 
 
 
자료: KB국민은행, 부동산114  자료: KB국민은행, 부동산114...
메타 데이터: {'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'moddate': '2024-03-04T15:30:01+09:00', 'creator': 'Microsoft® Word 2016', 'producer': 'Microsoft® Word 2016', 'total_pages': 84, 'author': '손은경', 'title': 'Morning Meeting', 'page_label': '25', 'creationdate': '2024-03-04T15:30:01+09:00', 'page': 24}
--------------------


In [31]:
# similarity_search_with_relevance_scores 메서드 사용
query = '수도권 주택 매매 전망'
top_three_docs = vectordb.similarity_search_with_relevance_scores(query, k=3)
for i,doc in enumerate(top_three_docs, 1): # 시작 번호 1
    print(f"문서 {i}:")
    print(f"유사 점수 {doc[1]}:")
    print(f"내용: {doc[0].page_content[:150]}...")
    print(f"메타 데이터: {doc[0].metadata}")
    print('-'*20)

문서 1:
유사 점수 0.844632892222533:
내용: 8 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
실 등에 따른 주택 경기 불안을 이유로 매매를 망설이며 시세 대비 저렴한 매물에만 관심을 보였다. 결
국 매도자와 매수자 간 희망가격 차이로 인한 매매 거래 위축 현상은 2023년 거래 침체의 가...
메타 데이터: {'page_label': '15', 'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'moddate': '2024-03-04T15:30:01+09:00', 'creationdate': '2024-03-04T15:30:01+09:00', 'author': '손은경', 'title': 'Morning Meeting', 'total_pages': 84, 'creator': 'Microsoft® Word 2016', 'page': 14, 'producer': 'Microsoft® Word 2016'}
--------------------
문서 2:
유사 점수 0.8328422165212013:
내용: 18 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
그림Ⅰ-30. 수도권 입주물량과 전세가격 변동률 추이  그림Ⅰ-31. 기타지방 입주물량과 전세가격 변동률 추이 
 
 
 
자료: KB국민은행, 부동산114  자료: KB국민은행, 부동산114...
메타 데이터: {'page': 24, 'producer': 'Microsoft® Word 2016', 'title': 'Morning Meeting', 'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'creator': 'Microsoft® Word 2016', 'page_label': '25', 'total_pages': 84, 'moddate': '2024-03-04T15:30:01+09:00', 'author': '손은경', 'creationdate': '2024-03-04T15:30:01+0

In [32]:
# FAISS
from langchain_community.vectorstores import FAISS

faiss_db = FAISS.from_documents(documents=splits, embedding=embedding_function)
print("문서 수: ", faiss_db.index.ntotal)

문서 수:  135


In [33]:
faiss_dir = './faissDB/'
faiss_db.save_local(faiss_dir)

new_db_faiss = FAISS.load_local(faiss_dir, OpenAIEmbeddings(), allow_dangerous_deserialization=True)
query = '수도권 주택 매매 전망'
docs = new_db_faiss.similarity_search(query)
for i, doc in enumerate(docs,1):
    print(f"문서 {i}:")
    print(f"내용: {doc.page_content[:150]}...")
    print(f"메타데이터: {doc.metadata}")
    print('--'*20)

문서 1:
내용: 8 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
실 등에 따른 주택 경기 불안을 이유로 매매를 망설이며 시세 대비 저렴한 매물에만 관심을 보였다. 결
국 매도자와 매수자 간 희망가격 차이로 인한 매매 거래 위축 현상은 2023년 거래 침체의 가...
메타데이터: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 14, 'page_label': '15'}
----------------------------------------
문서 2:
내용: 18 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
그림Ⅰ-30. 수도권 입주물량과 전세가격 변동률 추이  그림Ⅰ-31. 기타지방 입주물량과 전세가격 변동률 추이 
 
 
 
자료: KB국민은행, 부동산114  자료: KB국민은행, 부동산114...
메타데이터: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './Data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 24, 'page_label': '25'}
--------------------------

In [34]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

loader =PyPDFLoader('./Data/2024_KB_부동산_보고서_최종.pdf')
documents = loader.load() # PDF 파일에서 문서 로드

# 텍스트 분할 설정: 청크 크기와 겹침 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print("분할된 청크 수: ", len(chunks))

# 임베딩 생성 및 Chroma DB 생성
embedding_function = OpenAIEmbeddings()
persist_directory = './chromaDB/'
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding = embedding_function,
    persist_directory=persist_directory
)

print('문서 수: ', vectordb._collection.count())

분할된 청크 수:  135
문서 수:  270


In [35]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
retriever = vectordb.as_retriever(search_kwargs={"k":3}) # top-k

# 프롬프트 템플릿 설정 : 사용자 질문에 대한 답변을 생성하기 위한 템플릿
template = """당신은 KB 부동산 보고서 전문가입니다. 다음 정보를 바탕으로 사용자의 질문에 답변해주세요.
컨텍스트 : {context}
"""

prompt = ChatPromptTemplate.from_messages( # 프롬프트 템플릿 생성
    [
        ("system",template), # 모델의 전반적인 역할과 행동 지침을 설정
        ("placeholder","{chat_history}"), # 대화 기록용으로 이전 대화 내용을 삽입
        ("human", "{question}") # 사용자의 질문이나 입력을 나타냄
    ]
)
model = ChatOpenAI(model_name='gpt-4o-mini', temperature=0) 
print(prompt.format(context="컨텍스트 예시", chat_history=["대화 기록 예시1", "대화 기록 예시2"], question='질문 예시'))

System: 당신은 KB 부동산 보고서 전문가입니다. 다음 정보를 바탕으로 사용자의 질문에 답변해주세요.
컨텍스트 : 컨텍스트 예시

Human: 대화 기록 예시1
Human: 대화 기록 예시2
Human: 질문 예시


In [ ]:
# 문서 형식 변환 함수 정의
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs) # 문서를 줄바꿈으로 연결

# 체인 구성: 검색한 문서를 프롬프트에 연결하고 모델을 통해 응답 생성
chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x['question']))
    ) # 검색된 문서를 연결하여 전달
    | prompt
    | model
    | StrOutputParser() # 결과를 문자열로 변환
)

# 대화 기록을 유지하기 위한 메모리 설정
chat_history = ChatMessageHistory()
chain_with_memory = RunnableWithMessageHistory(
    chain,
    lambda session_id : chat_history, # 세션 ID별 대화 기록 생성
    input_messages_key = "question", # invoke 입력에서 사용자의 쿼리가 있는 딕셔너리의 키는 "question"이다
    history_messages_key="chat_history" # 꺼낸 채팅 기록을 placeholder의 "chat_history" 부분에 넣어라
)

# 챗봇 실행 함수
def chat_with_bot():
    session_id = "user_session"
    print("KB 부동산 보고서 챗봇입니다. 질문해 주세요. (종료하려면 'quit' 입력)")
    while True:
        user_input = input("사용자: ")
        if user_input.lower() == 'quit':
            break

        # 사용자의 질문에 따라 chain_with_memory를 통해 응답 생성
        response = chain_with_memory.invoke(
            {"question":user_input},
            {"configurable":{"session_id":session_id}}
        )

        print("챗봇: ", response)

c:\Users\Dohy\Desktop\dohy\RAG_master\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
#ngrok.set_auth_token(API_KEY) 
# 터미널에서 !streamlit run ./chatbot.py

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501) # Streamlit 기본 포트
print("앱 접속 URL: ", public_url)


앱 접속 URL:  NgrokTunnel: "https://makeover-doornail-clambake.ngrok-free.dev" -> "http://localhost:8501"
^C
